# Stage 1 — Topic Model

Generates embeddings, tunes UMAP and HDBSCAN parameters, trains the BERTopic model, and produces annotated outputs and annotation samples.

**Runtime:** Google Colab + Google Drive  
**Storage:** All inputs and outputs are read from / written to a project folder on Google Drive.  
**Library:** [`multilingual-topic-modeling`](https://github.com/ay94/multilingual-topic-modeling)

**Inputs:** `cleaned_dataset.jsonl.gz` from Stage 0  
**Outputs:**
- `embeddings.json` — sentence embeddings
- `topic_model.bertopic` — saved BERTopic model
- `annotated_messages.jsonl.gz` — messages with topic assignments
- `topic_info.csv` — topic keywords and representative docs
- `topic_model_samples.csv` — annotation sample (10% per topic)

See [`docs/workflow/`](https://github.com/ay94/topic-modeling-recipes/tree/main/docs/workflow) for the full methodology.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
!pip install multilingual-topic-modeling --quiet

In [ ]:
import time
import numpy as np
import pandas as pd
import plotly.express as px
from collections import Counter

from umap import UMAP
from hdbscan import HDBSCAN
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer

from multilingual_topic import FileHandler
from multilingual_topic import ParameterTuner, TopicModel

## Configuration

In [ ]:
# ── Edit these ────────────────────────────────────────────────────────────────
DRIVE_FOLDER    = '/content/drive/MyDrive/YOUR_PROJECT/Topic Modelling Workflow'
EMBEDDING_MODEL = 'all-MiniLM-L6-v2'   # HuggingFace sentence-transformer model
TEXT_COL        = 'cleaned_text'
TOPIC_PREFIX    = 'topics'              # prefix for topic columns in output
LANG            = 'en'                  # 'en' adds English stopwords to vectorizer
# ──────────────────────────────────────────────────────────────────────────────

fh = FileHandler(DRIVE_FOLDER)

## 1. Load data

In [ ]:
dataset = pd.read_json(
    fh.create_filename('data/cleaned_dataset.jsonl.gz'),
    lines=True,
)
docs = dataset[TEXT_COL].values
print(f'Documents: {len(docs):,}')

## 2. Create embeddings

In [ ]:
sentence_model = SentenceTransformer(EMBEDDING_MODEL)
print(f'Max sequence length: {sentence_model.max_seq_length}')

In [ ]:
embeddings = sentence_model.encode(docs, show_progress_bar=True)
print(f'Embeddings shape: {embeddings.shape}')

In [ ]:
# Save embeddings — loading from disk is faster than re-encoding on rerun
fh.save_embeddings(f'outputs/embeddings-{EMBEDDING_MODEL}.json', docs, embeddings)

# To reload later:
# docs, embeddings = fh.load_embeddings(f'outputs/embeddings-{EMBEDDING_MODEL}.json')

## 3. Parameter tuning

Use 2D UMAP to visualise the embedding space and get an intuition for cluster structure before committing to parameters. Then apply the chosen parameters with 5D UMAP (the standard BERTopic setup) and check the resulting topic count and -1 proportion.

See [`docs/workflow/`](https://github.com/ay94/topic-modeling-recipes/tree/main/docs/workflow) for guidance on UMAP and HDBSCAN parameter choices.

In [ ]:
# 2D visualisation
standard_embedding = UMAP(
    n_neighbors=15, n_components=2, min_dist=0.0, random_state=1, metric='cosine'
).fit_transform(embeddings)

fig = px.scatter(
    x=standard_embedding[:, 0], y=standard_embedding[:, 1],
    title='2D UMAP — embedding structure'
)
fig.show()

In [ ]:
# ── Edit these to tune ────────────────────────────────────────────────────────
N_NEIGHBORS      = 15
N_COMPONENTS     = 5
MIN_DIST         = 0.0
MIN_CLUSTER_SIZE = 30
# ──────────────────────────────────────────────────────────────────────────────

umap_model    = UMAP(n_neighbors=N_NEIGHBORS, n_components=N_COMPONENTS,
                     min_dist=MIN_DIST, random_state=1, metric='cosine')
hdbscan_model = HDBSCAN(min_cluster_size=MIN_CLUSTER_SIZE,
                         metric='euclidean', prediction_data=True)

start = time.time()
umap_embeddings = umap_model.fit_transform(embeddings)
hdbscan_labels  = hdbscan_model.fit_predict(umap_embeddings)
print(f'Done in {time.time()-start:.0f}s')

topic_counts = pd.DataFrame(Counter(hdbscan_labels).items(), columns=['topic', 'count'])
topic_counts['pct'] = (topic_counts['count'] / topic_counts['count'].sum() * 100).round(1)
print(f"Topics: {(topic_counts['topic'] >= 0).sum()}  |  -1 outliers: {topic_counts.loc[topic_counts['topic']==-1,'pct'].values[0]:.1f}%")
topic_counts.sort_values('count', ascending=False)

In [ ]:
# Visualise with HDBSCAN labels overlaid
umap_2d = pd.DataFrame(standard_embedding, columns=['x', 'y'])
umap_2d['topic'] = [f'topic-{t}' for t in hdbscan_labels]
fig = px.scatter(umap_2d, x='x', y='y', color='topic', title='2D UMAP — HDBSCAN topics')
fig.show()

## 4. Train BERTopic model

In [ ]:
vectorizer_model = CountVectorizer(stop_words='english' if LANG == 'en' else None, min_df=2)

umap_model    = UMAP(n_neighbors=N_NEIGHBORS, n_components=N_COMPONENTS,
                     min_dist=MIN_DIST, random_state=1, metric='cosine')
hdbscan_model = HDBSCAN(min_cluster_size=MIN_CLUSTER_SIZE,
                         metric='euclidean', prediction_data=True)

topic_model = BERTopic(
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    calculate_probabilities=True,
    verbose=True,
)

start = time.time()
topics, probs = topic_model.fit_transform(documents=docs, embeddings=embeddings)
print(f'Done in {time.time()-start:.0f}s')

topic_info = topic_model.get_topic_info()
topic_info

## 5. Annotate and save

In [ ]:
dataset[f'{TOPIC_PREFIX}'] = topics
dataset[f'{TOPIC_PREFIX}_outlier'] = [t == -1 for t in topics]

# Add topic keyword summary to each row
topic_name_map = {
    row['Topic']: ' / '.join([w for w, _ in topic_model.get_topic(row['Topic'])][:5])
    for _, row in topic_info.iterrows() if row['Topic'] != -1
}
dataset[f'{TOPIC_PREFIX}_name'] = dataset[TOPIC_PREFIX].map(topic_name_map).fillna('outlier')

print(f"Outliers: {dataset[f'{TOPIC_PREFIX}_outlier'].sum():,} ({dataset[f'{TOPIC_PREFIX}_outlier'].mean()*100:.1f}%)")

In [ ]:
# Save annotated messages
dataset.to_json(
    fh.create_filename('outputs/annotated_messages.jsonl.gz'),
    orient='records', lines=True,
)

# Save topic info
topic_info.to_csv(fh.create_filename('outputs/topic_info.csv'), index=False)

# Save BERTopic model
topic_model.save(fh.create_filename('outputs/topic_model.bertopic'))

print('Saved.')

## 6. Generate annotation sample

10% per topic (minimum 20 messages). This is the sample annotators will review to produce the topic theme map.

In [ ]:
inliers = dataset[~dataset[f'{TOPIC_PREFIX}_outlier']]
samples = []
for tp, tp_df in inliers.groupby(TOPIC_PREFIX):
    n = max(20, int(len(tp_df) * 0.1))
    samples.append(tp_df.sample(n=min(n, len(tp_df)), random_state=1))

sample_df = pd.concat(samples)
sample_df.to_csv(fh.create_filename('outputs/topic_model_samples.csv'), index=False)
print(f'Sample: {len(sample_df):,} rows across {inliers[TOPIC_PREFIX].nunique()} topics')

## 7. Propagate labels to duplicates

Transform duplicate messages using the trained model so all data is labelled, not just the unique set.

In [ ]:
duplicates = pd.read_json(
    fh.create_filename('data/cleaned_dataset_duplicates.jsonl.gz'),
    lines=True,
)

dup_docs       = duplicates[TEXT_COL].values
dup_embeddings = sentence_model.encode(dup_docs, show_progress_bar=True)
dup_topics, _  = topic_model.transform(documents=dup_docs, embeddings=dup_embeddings)

duplicates[TOPIC_PREFIX] = dup_topics
duplicates[f'{TOPIC_PREFIX}_outlier'] = [t == -1 for t in dup_topics]
duplicates[f'{TOPIC_PREFIX}_name'] = duplicates[TOPIC_PREFIX].map(topic_name_map).fillna('outlier')

duplicates.to_json(
    fh.create_filename('outputs/annotated_duplicates.jsonl.gz'),
    orient='records', lines=True,
)
print(f'Propagated to {len(duplicates):,} duplicate records.')